# What Makes a GitHub Repository Popular?
## A simple data science project in R

**Model:** Simple Linear Regression  
**Dataset:** GitHub Trending Repositories Dataset 2025 (provided CSV files)

### Marking scheme alignment
- **2 marks:** problem relevance, problem statement, objectives and data source
- **3 marks:** exploratory data analysis (EDA)
- **5 marks:** regression demo, evaluation and report

> **Important dataset finding:** `contributors_count` is 0 for all 1,587 rows, so it cannot explain differences and is excluded from the model. This notebook uses `forks` as the single predictor and predicts `stars_period`. The model describes association, not causation. The data only includes repositories captured as trending, so results may not generalize to all GitHub repositories.

Each code section has simple comments explaining what it does.

## 1. Problem statement, relevance and objectives

**Problem statement:** Can the number of forks help estimate how many stars a GitHub repository gained during the recorded trending period?

**Why it matters:** Stars and forks are common visible signals of developer interest. Understanding how they relate to short-term star gains can help describe engagement patterns in trending repositories.

**Objectives**
1. Inspect and clean the repository dataset.
2. Explore languages, stars, forks and trending-period star gains.
3. Fit a simple Linear Regression model in R.
4. Evaluate predictions on held-out test data.

**Data source:** [Kaggle — GitHub Trending Repositories Dataset 2025](https://www.kaggle.com/datasets/mihikaajayjadhav/github-trending-repositories-dataset-2025/data)

**Target:** `stars_period`  
**Predictor:** `forks`

In [ ]:
# Load the dataset from the data folder beside this notebook.
repos <- read.csv("data/github_trending_repos.csv", stringsAsFactors = FALSE)

# Show the dataset size and available columns.
cat("Rows:", nrow(repos), "\n")
cat("Columns:", ncol(repos), "\n")
names(repos)

## 2. Data inspection and cleaning

In [ ]:
# Preview the first six repositories to understand the data.
head(repos)

# Check the data types and basic structure of each column.
str(repos)

# Count missing values in each column.
colSums(is.na(repos))

In [ ]:
# Convert the columns used in analysis to numeric values.
repos$stars <- as.numeric(gsub(",", "", repos$stars))
repos$forks <- as.numeric(gsub(",", "", repos$forks))
repos$stars_period <- as.numeric(gsub(",", "", repos$stars_period))
repos$contributors_count <- as.numeric(gsub(",", "", repos$contributors_count))

# Replace blank descriptions with an empty string for clean reporting.
repos$description[is.na(repos$description)] <- ""

# Remove rows missing the target or the selected predictor.
repos <- repos[complete.cases(repos[, c("forks", "stars_period")]), ]

# Keep rows with non-negative counts.
repos <- repos[repos$forks >= 0 & repos$stars_period >= 0, ]

cat("Rows after cleaning:", nrow(repos), "\n")

## 3. Exploratory Data Analysis (EDA) — 3 marks

In [ ]:
# Summary statistics describe the range and typical values of the metrics.
summary(repos[, c("stars", "forks", "stars_period", "contributors_count")])

In [ ]:
# Count repositories by primary programming language.
language_counts <- sort(table(repos$language), decreasing = TRUE)

# Display the ten most frequent languages.
head(language_counts, 10)

# Plot the ten most frequent languages to compare their counts.
barplot(head(language_counts, 10),
        main = "Top 10 Languages in the Dataset",
        xlab = "Language",
        ylab = "Repository rows",
        las = 2,
        cex.names = 0.8)

In [ ]:
# Plot forks against stars gained during the trending period.
# Each point represents one repository row in the dataset.
plot(repos$forks, repos$stars_period,
     main = "Forks vs Stars Gained During Trending",
     xlab = "Number of forks",
     ylab = "Stars gained during trending period",
     pch = 19)

# Add a simple trend line to help visualize the overall relationship.
abline(lm(stars_period ~ forks, data = repos), lwd = 2, lty = 2)

In [ ]:
# Calculate correlations between the main numeric metrics.
# Correlation measures linear association, not cause and effect.
cor(repos[, c("stars", "forks", "stars_period", "contributors_count")],
    use = "complete.obs")

In [ ]:
# Check whether contributor_count varies in this dataset.
# If it has only one unique value, it cannot help explain popularity differences.
table(repos$contributors_count)
length(unique(repos$contributors_count))

### Initial dataset observations

- The CSV contains **1,587 rows and 13 columns**.
- `description` has 36 missing values; those descriptions are not required for this model.
- `contributors_count` is zero for every row, so it is not useful as a predictor in this dataset.
- Repository entries appear under different search-language/timeframe combinations. Treat the analysis as a description of the supplied dataset rather than a random sample of every GitHub repository.

## 4. Train a simple Linear Regression model — 5 marks

In [ ]:
# Keep only the target and one predictor to make the model easy to explain.
model_data <- repos[, c("forks", "stars_period")]

# Use log1p to reduce the effect of very large counts and zero values.
# The model predicts log(1 + stars_period) from log(1 + forks).
model_data$log_forks <- log1p(model_data$forks)
model_data$log_stars_period <- log1p(model_data$stars_period)

# Randomly split rows: 80% for training and 20% for testing.
set.seed(42)  # Use the same split each time the notebook runs.
train_index <- sample(seq_len(nrow(model_data)),
                      size = floor(0.8 * nrow(model_data)))

train_data <- model_data[train_index, ]
test_data <- model_data[-train_index, ]

# Fit the regression line using the training rows only.
model <- lm(log_stars_period ~ log_forks, data = train_data)

# Display the intercept, slope and model summary.
summary(model)

In [ ]:
# Predict log star gains for rows held out from training.
predicted_log <- predict(model, newdata = test_data)

# Convert predictions back to the original star-count scale.
predicted_stars <- pmax(0, expm1(predicted_log))
actual_stars <- test_data$stars_period

# Calculate MAE: average absolute prediction error in stars.
mae <- mean(abs(actual_stars - predicted_stars))

# Calculate RMSE: penalizes larger prediction errors more strongly.
rmse <- sqrt(mean((actual_stars - predicted_stars)^2))

# Calculate test R-squared against a baseline that predicts the test mean.
baseline_error <- sum((actual_stars - mean(actual_stars))^2)
test_r_squared <- if (baseline_error > 0) {
  1 - sum((actual_stars - predicted_stars)^2) / baseline_error
} else {
  NA_real_
}

# Print evaluation metrics. R-squared may be negative if the model
# performs worse than the mean-value baseline.
cat("Test rows:", nrow(test_data), "\n")
cat("MAE (stars):", round(mae, 2), "\n")
cat("RMSE (stars):", round(rmse, 2), "\n")
cat("Test R-squared:", round(test_r_squared, 3), "\n")

In [ ]:
# Compare actual and predicted values in a small table.
results <- data.frame(
  actual_stars_period = actual_stars,
  predicted_stars_period = round(predicted_stars, 2),
  absolute_error = round(abs(actual_stars - predicted_stars), 2)
)

head(results, 10)

In [ ]:
# Plot actual values against predictions.
# Points nearer the diagonal line indicate more accurate predictions.
plot(actual_stars, predicted_stars,
     main = "Actual vs Predicted Stars Gained",
     xlab = "Actual stars gained",
     ylab = "Predicted stars gained",
     pch = 19)

# Add the perfect-prediction reference line.
abline(a = 0, b = 1, lty = 2, lwd = 2)

In [ ]:
# Save predictions so they can be used in the report.
write.csv(results, "regression_predictions.csv", row.names = FALSE)

# Save a language summary for the report.
language_summary <- aggregate(
  cbind(stars, forks, stars_period) ~ language,
  data = repos,
  FUN = mean
)
language_summary$repository_rows <- as.vector(table(repos$language)[language_summary$language])
write.csv(language_summary, "language_summary_from_notebook.csv", row.names = FALSE)

cat("Saved regression_predictions.csv and language_summary_from_notebook.csv\n")

## 5. Report and conclusion template

After running all cells, use the actual outputs above to complete these points:

1. **EDA:** State which languages have the most rows and describe the observed distributions of stars, forks and trending-period star gains.
2. **Relationship:** Use the scatter plot and regression slope to describe whether forks are positively or negatively associated with star gains.
3. **Model performance:** Report the actual MAE, RMSE and test R² printed by the notebook.
4. **Limitations:** The contributor count is constant at zero; the dataset consists of repositories captured as trending; repeated repositories may appear under multiple collection contexts; correlation does not prove causation; and this simple model may not predict all repositories well.

**Do not write that the model is accurate unless the test metrics support that conclusion.**